In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import tensorflow_hub as hub

# =========================
# 📂 PATHS (عدلهم لو محتاج)
# =========================
train_dir = "D:/New folder (2)/project work/train"
valid_dir = "D:/New folder (2)/project work/valid"
test_dir  = "D:/New folder (2)/project work/test"

# =========================
# ⚙️ PARAMETERS
# =========================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30

# =========================
# 📊 DATA GENERATORS
# =========================
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

valid_gen = ImageDataGenerator(rescale=1./255)
test_gen  = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_data = valid_gen.flow_from_directory(
    valid_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_data.num_classes
print("عدد الكلاسات:", num_classes)

# =========================
# 🧠 MODEL
# =========================
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# =========================
# 1️⃣ CNN
# =========================
x1 = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
x1 = layers.MaxPooling2D()(x1)

x1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x1)
x1 = layers.MaxPooling2D()(x1)

x1 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x1)
x1 = layers.GlobalAveragePooling2D()(x1)

# =========================
# 2️⃣ EfficientNet
# =========================
eff_input = preprocess_input(inputs)

eff_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_tensor=eff_input
)

for layer in eff_model.layers:
    layer.trainable = False

x2 = eff_model.output
x2 = layers.GlobalAveragePooling2D()(x2)

# =========================
# 3️⃣ ViT
# =========================
vit_layer = hub.KerasLayer(
    "https://tfhub.dev/sayakpaul/vit_b16_fe/1",
    trainable=False
)

x3 = vit_layer(inputs)

# =========================
# 🔥 FUSION
# =========================
combined = layers.Concatenate()([x1, x2, x3])

# =========================
# CLASSIFIER
# =========================
x = layers.BatchNormalization()(combined)

x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=inputs, outputs=outputs)

# =========================
# COMPILE
# =========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =========================
# 📉 CALLBACKS
# =========================
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model.h5", save_best_only=True),
    ReduceLROnPlateau(patience=3, factor=0.3)
]

# =========================
# 🚀 TRAIN
# =========================
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=EPOCHS,
    callbacks=callbacks
)

# =========================
# 📊 EVALUATE
# =========================
loss, acc = model.evaluate(test_data)
print(f"Test Accuracy: {acc*100:.2f}%")

# =========================
# 💾 SAVE MODEL
# =========================
model.save("final_triple_model.h5")

print("تم حفظ الموديل 🔥") 


✅ GPU Memory Growth Enabled
Found 3698 files belonging to 6 classes.
Found 902 files belonging to 6 classes.
⏳ Calculating Class Weights for balancing...
📊 Calculated Class Weights: {0: 0.7175009701202949, 1: 1.2786998616874135, 2: 1.0087288597926896, 3: 2.282716049382716, 4: 0.8997566909975669, 5: 0.7791824694479562}
🚀 Starting Balanced Training...
Epoch 1/30
463/463 [==============================] - ETA: 0s - loss: 1.3524 - accuracy: 0.5776
Epoch 1: val_accuracy improved from -inf to 0.77494, saving model to final_skin_model_balanced.weights.h5
463/463 [==============================] - 125s 229ms/step - loss: 1.3524 - accuracy: 0.5776 - val_loss: 0.9661 - val_accuracy: 0.7749 - lr: 1.0000e-04
Epoch 2/30
463/463 [==============================] - ETA: 0s - loss: 1.0616 - accuracy: 0.7207
Epoch 2: val_accuracy improved from 0.77494 to 0.82373, saving model to final_skin_model_balanced.weights.h5
463/463 [==============================] - 104s 225ms/step - loss: 1.0616 - accuracy: 0.7